# Modelos de riesgo de mercado

Cuánto se puede perder, con qué probabilidad, y —sobre todo— si el modelo que lo
predice acierta.

Convención de signos: todas las medidas de riesgo se devuelven como **pérdida
positiva**. Un VaR al 99 % de `0.031` significa que se espera perder más del 3,1 %
sólo el 1 % de los días.

Como en el primer notebook, si no hay acceso a Yahoo Finance se trabaja con datos
simulados y se avisa por pantalla.

In [ ]:
import warnings

import numpy as np
import pandas as pd
from scipy import stats

from icm import graficos
from icm.cartera import contraste_jarque_bera, curtosis, asimetria
from icm.datos import cargar_precios, matriz_covarianzas, rendimientos, rendimientos_cartera
from icm.riesgo import (
    ajustar_copula_t,
    ajustar_garch,
    backtest_var,
    bootstrap_historico,
    comparar_copulas,
    comparar_metodos_var,
    comparar_modelos_volatilidad,
    dependencia_cola_empirica,
    es_montecarlo,
    maxima_caida,
    resumen_simulacion,
    serie_caidas,
    serie_var_movil,
    simular_cartera,
    tabla_caidas,
    tabla_medidas,
    var_montecarlo,
    volatilidad_ewma,
    volatilidad_movil,
    zona_basilea,
)

graficos.aplicar_estilo()
pd.set_option("display.precision", 4)
pd.set_option("display.width", 140)

In [ ]:
TICKERS = ["AAPL", "MSFT", "JNJ", "XOM", "JPM"]
VALOR_CARTERA = 1_000_000

with warnings.catch_warnings(record=True) as avisos:
    warnings.simplefilter("always")
    precios = cargar_precios(TICKERS, "2015-01-01", "2024-12-31", respaldo_simulado=True)

DATOS_SIMULADOS = any("SIMULADOS" in str(a.message) for a in avisos)
if DATOS_SIMULADOS:
    print("!! Sin acceso a Yahoo Finance: se trabaja con DATOS SIMULADOS.\n")
else:
    print(f"Datos reales de Yahoo Finance: {len(precios)} sesiones.\n")

rend = rendimientos(precios, "log")
pesos = pd.Series(1.0 / len(TICKERS), index=rend.columns)
cartera = rendimientos_cartera(rend, pesos)
print(f"Cartera equiponderada de {len(TICKERS)} activos, {len(cartera)} sesiones.")

## 1. La pregunta previa: ¿son normales los rendimientos?

Todo el VaR paramétrico clásico descansa sobre la normalidad. Si los datos no son
normales, el modelo no es conservador: es directamente incorrecto, y lo es
precisamente en la cola, que es donde importa.

In [ ]:
import matplotlib.pyplot as plt

figura, ax = plt.subplots(figsize=(9, 4.5))
ax.hist(cartera, bins=120, density=True, color=graficos.PALETA[0], alpha=0.75,
        linewidth=0, label="Rendimientos observados")

rejilla = np.linspace(cartera.min(), cartera.max(), 500)
ax.plot(rejilla, stats.norm.pdf(rejilla, cartera.mean(), cartera.std(ddof=1)),
        color=graficos.PALETA[1], linewidth=2, label="Normal con la misma media y varianza")

ax.set_xlabel("Rendimiento diario")
ax.set_ylabel("Densidad")
ax.set_title("La normal se queda corta justo donde importa")
ax.legend(loc="upper left")
ax.grid(axis="x", visible=False)
plt.show()

prueba = contraste_jarque_bera(cartera)
print(f"Asimetría:          {asimetria(cartera):+.3f}   (0 si fuese normal)")
print(f"Exceso de curtosis: {curtosis(cartera):+.3f}   (0 si fuese normal)")
print(f"Jarque-Bera p-valor: {prueba['p_valor']:.2e}  -> "
      f"{'no se rechaza' if prueba['normal_al_5pct'] else 'SE RECHAZA'} la normalidad")

## 2. Cuatro formas de calcular el mismo número

- **Histórico**: el cuantil empírico. No supone nada, pero no sabe nada de pérdidas
  mayores que la peor observada.
- **Paramétrico normal**: fórmula cerrada. Rápido y, con colas gruesas, optimista.
- **Paramétrico t**: mismo enfoque con colas gruesas ajustadas por verosimilitud.
- **Cornish-Fisher**: corrige el cuantil normal con la asimetría y la curtosis
  observadas. Un término medio barato.

El Expected Shortfall (ES) es la pérdida media *dentro* de la cola. A diferencia
del VaR, es una medida coherente —diversificar nunca puede empeorarlo— y es la que
Basilea III adoptó en sustitución del VaR.

In [ ]:
medidas = tabla_medidas(cartera, niveles=(0.95, 0.99), valor_cartera=VALOR_CARTERA)
medidas.round(5)

In [ ]:
al_99 = medidas.xs(0.99, level="nivel")
diferencia = al_99.loc["historico", "var"] / al_99.loc["parametrico_normal", "var"] - 1
print(f"El VaR histórico al 99 % es un {diferencia:+.1%} respecto al paramétrico normal.")
print(f"En euros, sobre una cartera de {VALOR_CARTERA:,.0f}: "
      f"{al_99.loc['historico', 'var_monetario']:,.0f} frente a "
      f"{al_99.loc['parametrico_normal', 'var_monetario']:,.0f}.")

## 3. La volatilidad no es constante

Los mercados alternan rachas tranquilas y turbulentas. Un VaR que use la
volatilidad media de los últimos años reacciona tarde y mal.

Tres estimadores, de menos a más sofisticado:

- **Media móvil**: fácil de explicar, pero cuando un día extremo sale de la ventana
  la volatilidad cae de golpe sin que haya pasado nada en el mercado.
- **EWMA** (RiskMetrics): los días pesan `(1-λ)·λ^k`, así que la influencia se
  desvanece suavemente en lugar de desaparecer.
- **GARCH**: modeliza explícitamente la dinámica de la varianza y permite prever.

In [ ]:
ajuste = ajustar_garch(cartera, "gjr", distribucion="t")

figura, ax = graficos.series_temporales(
    {
        "Media móvil (21 d)": volatilidad_movil(cartera, 21, anualizar=True),
        "EWMA (λ = 0,94)": volatilidad_ewma(cartera, anualizar=True),
        "GJR-GARCH(1,1)": ajuste.volatilidad_condicional(anualizar=True),
    },
    etiqueta_y="Volatilidad anualizada",
    porcentaje=True,
)
ax.set_title("Tres lecturas de la misma volatilidad cambiante");

In [ ]:
print(f"Persistencia estimada: {ajuste.persistencia:.3f}")
print("Cuanto más cerca de 1, más tardan en disiparse los choques de volatilidad.\n")
comparar_modelos_volatilidad(cartera, distribucion="t").round(3)

El GJR añade un término que deja que las caídas eleven más la volatilidad que las
subidas: es el efecto apalancamiento, muy marcado en renta variable. Si mejora el
BIC frente al GARCH simétrico, esa asimetría está en los datos.

## 4. Lo que el inversor percibe: las caídas

La volatilidad es una medida simétrica y abstracta. Lo que de verdad duele es
cuánto se ha llegado a perder desde el último máximo y cuánto se ha tardado en
recuperarlo.

In [ ]:
figura, ax = graficos.caidas(serie_caidas(cartera))
ax.set_title("Distancia al máximo histórico en cada momento");

In [ ]:
peores = tabla_caidas(cartera, n=5)
peores.assign(caida=lambda t: (t["caida"] * 100).round(1))

## 5. Simulación: de una cifra a una distribución

El VaR resume el riesgo en un número. La simulación da la distribución completa, y
con ella se pueden responder preguntas que un cuantil no responde: qué
probabilidad hay de perder más de un 10 % en un mes.

Dos enfoques complementarios:

- **Monte Carlo** con t multivariante: supone una forma para la distribución, pero
  permite extrapolar más allá de lo observado.
- **Bootstrap por bloques**: remuestrea tramos consecutivos de la historia real, de
  modo que conserva el agrupamiento de volatilidad sin suponer nada.

In [ ]:
HORIZONTE = 21   # un mes bursátil

mu_diaria = rend.mean()
cov_diaria = matriz_covarianzas(rend)

montecarlo = simular_cartera(
    mu_diaria, cov_diaria, pesos, n_simulaciones=100_000,
    horizonte=HORIZONTE, distribucion="t", grados_libertad=5.0, semilla=42,
)
figura, ax = graficos.distribucion_simulada(
    montecarlo, var=var_montecarlo(montecarlo, 0.99), es=es_montecarlo(montecarlo, 0.99)
)
ax.set_title(f"Monte Carlo: rendimiento de la cartera a {HORIZONTE} sesiones");

In [ ]:
bloques = bootstrap_historico(
    rend, pesos, n_simulaciones=100_000, horizonte=HORIZONTE, tamano_bloque=5, semilla=42
)
comparacion_simulacion = pd.DataFrame(
    {
        "Monte Carlo (t)": resumen_simulacion(montecarlo, valor_cartera=VALOR_CARTERA),
        "Bootstrap por bloques": resumen_simulacion(bloques, valor_cartera=VALOR_CARTERA),
    }
)
comparacion_simulacion.loc[
    ["media", "desviacion", "minimo", "prob_perdida", "var_99%", "es_99%",
     "prob_perder_mas_10%", "var_99%_monetario"]
].round(4)

## 6. El error que costó caro en 2008: ignorar la dependencia de cola

La matriz de correlación resume la dependencia **media**. No dice nada de lo que
pasa en los extremos, y es ahí donde se decide si una cartera aguanta.

La cópula separa el problema en dos: por un lado el comportamiento individual de
cada activo, por otro la estructura de dependencia. Y la elección de cópula tiene
una consecuencia que no es un detalle técnico:

- La **cópula gaussiana** tiene dependencia de cola **cero**. Por mucha correlación
  que se le ponga, la probabilidad de que dos activos sufran su peor día a la vez
  tiende a cero.
- La **cópula t** sí la tiene, gobernada por sus grados de libertad.

In [ ]:
copula_t = ajustar_copula_t(rend)
print(f"Grados de libertad estimados: {copula_t.grados_libertad:.2f}")
print(f"Dependencia de cola implícita (cópula t):        {copula_t.dependencia_cola():.3f}")
print(f"Dependencia de cola implícita (cópula gaussiana): 0.000\n")
print("Frecuencia observada con que dos activos entran juntos en su peor 5 %:")
dependencia_cola_empirica(rend, cuantil=0.05).round(3)

In [ ]:
tabla_copulas = comparar_copulas(rend, pesos, nivel=0.99, n_simulaciones=100_000)
tabla_copulas.round(4)

In [ ]:
infravaloracion = tabla_copulas.loc["t", "es_99%"] / tabla_copulas.loc["gaussiana", "es_99%"] - 1
print(f"La cópula gaussiana infravalora el ES al 99 % en un {infravaloracion:.1%}.")
print("La diferencia en AIC/BIC dice cuál de las dos describe mejor estos datos.")

## 7. La prueba de fuego: ¿acierta el modelo?

Un modelo de riesgo no vale por lo elegante que sea, sino por si acierta. Un VaR al
99 % debe superarse aproximadamente un 1 % de los días, **y las excepciones deben
aparecer dispersas**. Si se concentran en una semana, el modelo no reacciona a los
cambios de régimen aunque el número total cuadre.

- **Kupiec** contrasta cuántas excepciones hay.
- **Christoffersen** contrasta si están agrupadas.
- El **semáforo de Basilea** traduce el número de excepciones en un recargo de
  capital.

Todo el VaR se estima aquí con ventana móvil y **sólo con información anterior a
cada fecha**. Es la parte delicada: un backtest que deje entrar el dato del propio
día da resultados excelentes y completamente falsos.

In [ ]:
NIVEL, VENTANA = 0.99, 250

resultados = comparar_metodos_var(
    cartera, nivel=NIVEL, ventana=VENTANA,
    metodos=("historico", "parametrico_normal", "parametrico_t", "cornish_fisher", "ewma"),
)
resultados.set_index("metodo").round(4)

In [ ]:
validos = resultados[resultados["modelo_valido"]]["metodo"].tolist()
rechazados = resultados[~resultados["modelo_valido"]]["metodo"].tolist()
print(f"Superan el contraste conjunto: {', '.join(validos) if validos else 'ninguno'}")
print(f"Se rechazan:                   {', '.join(rechazados) if rechazados else 'ninguno'}\n")

for _, fila in resultados.iterrows():
    print(f"{fila['metodo']:>20}: {int(fila['excepciones']):>3} excepciones "
          f"(esperadas {fila['excepciones_esperadas']:.1f}), zona {fila['zona_basilea']}")

In [ ]:
peor = resultados.loc[resultados["p_condicional"].idxmin(), "metodo"]
var_peor = serie_var_movil(cartera, NIVEL, VENTANA, peor)
figura, ax = graficos.backtest(cartera, var_peor)
ax.set_title(f"Backtest del método '{peor}': el que peor supera el contraste");

In [ ]:
mejor = resultados.loc[resultados["p_condicional"].idxmax(), "metodo"]
detalle = backtest_var(cartera, NIVEL, VENTANA, mejor)
var_mejor = detalle.var
figura, ax = graficos.backtest(cartera, var_mejor, detalle.excepciones)
ax.set_title(f"Backtest del método '{mejor}': el que mejor lo supera")

print(f"Kupiec        p = {detalle.kupiec['p_valor']:.3f}")
print(f"Independencia p = {detalle.independencia['p_valor']:.3f}")
print(f"Conjunto      p = {detalle.condicional['p_valor']:.3f}")
print(f"Zona de Basilea: {detalle.basilea['zona']} "
      f"(recargo {detalle.basilea['recargo_multiplicador']:.2f})")

## 8. Qué llevarse de aquí

1. **Los rendimientos no son normales**, y el VaR paramétrico normal falla
   precisamente en la cola. La sección 1 lo mide y la 7 lo confirma contando
   excepciones.
2. **La volatilidad cambia de régimen.** Un modelo condicional (EWMA, GARCH) sube el
   VaR el día después de una sesión mala; uno histórico tarda tantos días como
   tenga la ventana.
3. **La dependencia de cola no es un refinamiento académico.** Elegir cópula
   gaussiana en lugar de t cambia el ES al 99 % en una cifra que se ve en la tabla
   de la sección 6.
4. **Contrastar el modelo no es opcional.** Kupiec y Christoffersen responden dos
   preguntas distintas, y un modelo puede pasar el primero y fallar el segundo: el
   número de excepciones cuadra, pero llegan todas juntas.
5. Y una advertencia de método: el VaR dice dónde está el umbral, no lo que hay
   detrás. Por eso el ES, que sí mira dentro de la cola, es la medida que ha
   acabado imponiéndose en la regulación.